# Leakage audit of T-PAL's data split

This notebook measures how many of T-PAL's test images have a near-copy or a same-patient image in
its training set. It runs `leakage_audit.py`, which is embedded below, on the Kaggle *Brain Tumor MRI
Dataset* (the dataset the T-PAL notebook used). It then turns the results into tables and figures
for the paper.

**What it reports**
- How many official *Testing* images, and how many T-PAL validation and test images, have a
  "family member" in training. A family member is an exact copy, a flipped or rotated near-copy, or
  (with the figshare step) a slice from the same patient.
- How those numbers change with the near-duplicate threshold.
- Example pairs, so you can check by eye that they really are copies.
- Families whose images carry different labels.
- Whether file details alone (image size, file size, format; no anatomy) predict the label.
- Leakage-free 5-fold splits, grouped by family, for re-training.

**How to run it on Kaggle**
1. **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. **Add Input**: search for `masoudnickparvar/brain-tumor-mri-dataset` and add it. Use the same
   dataset version as the T-PAL notebook, and write down the version number shown in the Input panel.
3. No GPU is needed. Leave the accelerator off.
4. *Optional, for patient IDs and tumour masks:* set `USE_FIGSHARE = True` in the next cell and turn
   on **Settings → Internet**. The notebook downloads the original figshare files (about 880 MB) and
   checks their MD5 sums.
5. **Run All.** Results go to `/kaggle/working/audit` and are zipped as `/kaggle/working/audit.zip`.
   Download them from the Output panel.

T-PAL's split depends on the order in which `os.walk` lists the folders. That order is only
guaranteed on Kaggle's own copy of the data, which is why the audit runs here. Step 2 checks that the
listing matches what the T-PAL notebook printed.

In [ ]:
import os, sys, json, glob, subprocess

USE_FIGSHARE = False   # True: download Cheng et al.'s figshare .mat files (~880 MB; needs Settings -> Internet on)
FIG_DIR = os.environ.get("AUDIT_FIGSHARE_DIR")  # or set this to an attached Kaggle dataset holding the .mat files
NCC = 0.98             # NCC needed to call two images near-duplicates (Vangala et al., arXiv:2610.00421, used 0.98)
NCC_FLOOR = 0.90       # lowest threshold kept for the sensitivity sweep
OUT = os.environ.get("AUDIT_OUT", "/kaggle/working/audit")
WORK = os.path.dirname(os.path.abspath(OUT))

## 1. Write the audit script
This is `code/leakage_audit.py` from the project, unchanged.

In [ ]:
%%writefile leakage_audit.py
"""
Leakage and shortcut audit for the Kaggle "masoudnickparvar/brain-tumor-mri-dataset" (or any
<root>/<split>/<class>/<image> folder).

  1. Rebuilds T-PAL's split exactly (os.walk order, SEED=42, pooled Training+Testing, 70/15/15
     stratified). Run it on the same Kaggle image as the notebook: os.walk order is not sorted, so
     the split is only reproducible on the same filesystem.
  2. Finds byte-identical and pixel-identical copies (MD5) and near-duplicates that survive flips
     and 90-degree rotations (64-bit DCT perceptual hash over the 8 dihedral transforms, confirmed
     by normalised cross-correlation).
  3. Optionally (--figshare) matches images to the original figshare .mat files (Cheng et al.) to
     recover patient IDs and tumour masks; images of one patient are merged into one family and the
     aligned masks are written to <out>/masks/ for the lesion-evidence loss.
  4. Groups images into families (union-find) and reports how many val/test images have a family
     member in train, per class, plus families whose members carry different labels.
  5. Runs a metadata-only shortcut probe: can image size, file size and colour mode alone (no
     anatomy) predict the label? Chance is 0.25 for 4 classes and 0.5 for tumour vs no tumour.
  6. Writes family-aware StratifiedGroupKFold folds for leakage-free experiments.

Usage on Kaggle:
  python leakage_audit.py --root /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset \
      --figshare /kaggle/input/<figshare-mat-folder> --out audit

Outputs in --out: images.csv (one row per image: md5s, family, patient, T-PAL split, fold, mask path),
pairs.csv (confirmed near-duplicate pairs, NCC >= --ncc), candidates.csv (every candidate pair with
NCC >= --ncc-floor, for threshold sweeps), summary.json, masks/*.png.
"""
from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os

import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
SEED = 42


def list_like_tpal(root, splits=("Training", "Testing")):
    """Same traversal as the T-PAL notebook (unsorted os.walk)."""
    rows = []
    for split in splits:
        for r, _, files in os.walk(os.path.join(root, split)):
            for f in files:
                if f.lower().endswith(EXTS):
                    rows.append({"image_path": os.path.join(r, f), "label": os.path.basename(r), "folder": split})
    return pd.DataFrame(rows)


def tpal_split(df):
    from sklearn.model_selection import train_test_split

    classes = sorted(df["label"].unique())
    y = df["label"].map({c: i for i, c in enumerate(classes)}).values
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=SEED)
    split = np.empty(len(df), dtype=object)
    split[tr], split[va], split[te] = "train", "val", "test"
    return split


def dihedral(a):
    """The 8 flips/rotations of a square array, in a fixed order (index t)."""
    out = []
    for k in range(4):
        r = np.rot90(a, k)
        out += [r, np.fliplr(r)]
    return out


def phash_bits(a32):
    low = dctn(a32, norm="ortho")[:8, :8]
    bits = (low > np.median(low)).flatten()
    return np.packbits(bits).view(">u8")[0]


def signature(gray_u8):
    """(8 dihedral pHashes, 64x64 float thumbnail) of a uint8 grayscale array."""
    im = Image.fromarray(gray_u8)
    a32 = np.asarray(im.resize((32, 32), Image.LANCZOS), dtype=np.float64)
    a64 = np.asarray(im.resize((64, 64), Image.BILINEAR), dtype=np.float64)
    return np.array([phash_bits(t) for t in dihedral(a32)], dtype=np.uint64), a64


def describe(path):
    raw = open(path, "rb").read()
    im = Image.open(path)
    meta = {"width": im.width, "height": im.height, "mode": im.mode, "fmt": im.format or "",
            "file_bytes": len(raw), "md5_file": hashlib.md5(raw).hexdigest()}
    g = np.asarray(im.convert("L"), dtype=np.uint8)
    meta["md5_pixels"] = hashlib.md5(g.tobytes()).hexdigest()
    meta["mean"], meta["std"] = float(g.mean()), float(g.std())
    meta["black_frac"] = float((g < 10).mean())
    b = max(1, int(0.05 * min(g.shape)))
    meta["border_mean"] = float(np.concatenate([g[:b].ravel(), g[-b:].ravel(), g[:, :b].ravel(), g[:, -b:].ravel()]).mean())
    hashes, a64 = signature(g)
    return meta, hashes, a64


def popcount64(x):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    x = x.copy()
    x -= (x >> np.uint64(1)) & np.uint64(0x5555555555555555)
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)


def hash_matches(HA, HB, radius, chunk=128):
    """Index pairs (a, b): identity hash of A[a] within `radius` bits of any dihedral hash of B[b]."""
    out = []
    for s in range(0, len(HA), chunk):
        d = popcount64(HA[s:s + chunk, 0][:, None, None] ^ HB[None]).min(-1)
        ia, ib = np.nonzero(d <= radius)
        out.append(np.stack([ia + s, ib], 1))
    return np.concatenate(out) if out else np.zeros((0, 2), int)


def ncc_best(a, b):
    """(max NCC over the 8 transforms of b, index of the best transform)."""
    za = (a - a.mean()) / (a.std() + 1e-8)
    s = [float((za * (t - t.mean()) / (t.std() + 1e-8)).mean()) for t in dihedral(b)]
    return max(s), int(np.argmax(s))


def zscore(thumbs):
    """(n, 64, 64) thumbnails -> float32, each image to zero mean and unit variance."""
    t = np.asarray(thumbs, dtype=np.float64)
    t = (t - t.mean((1, 2), keepdims=True)) / (t.std((1, 2), keepdims=True) + 1e-8)
    return t.astype(np.float32)


def ncc_pairs(ZA, ZB, pairs, chunk=4096, label=None):
    """Vectorised ncc_best for many pairs: ZA[i] against the 8 transforms of ZB[j].

    A flip or rotation only permutes pixels, so normalising before transforming gives the same NCC
    as ncc_best. Returns (best score, index of the best transform) per pair, in dihedral() order.
    """
    pairs = np.asarray(pairs, dtype=np.int64).reshape(-1, 2)
    best = np.empty(len(pairs), dtype=np.float32)
    arg = np.empty(len(pairs), dtype=np.int64)
    npx = ZA.shape[1] * ZA.shape[2]
    step = max(1, len(pairs) // 10)
    for s in range(0, len(pairs), chunk):
        p = pairs[s:s + chunk]
        a, b = ZA[p[:, 0]], ZB[p[:, 1]]
        scores = []
        for k in range(4):
            r = np.rot90(b, k, axes=(1, 2))
            scores += [(a * r).sum((1, 2)) / npx, (a * r[:, :, ::-1]).sum((1, 2)) / npx]
        scores = np.stack(scores, 1)
        best[s:s + chunk], arg[s:s + chunk] = scores.max(1), scores.argmax(1)
        if label and (s // step) != ((s + len(p)) // step):
            print(f"  {label}: {min(s + len(p), len(pairs)):,}/{len(pairs):,} pairs checked", flush=True)
    return best, arg


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, i):
        while self.p[i] != i:
            self.p[i] = self.p[self.p[i]]
            i = self.p[i]
        return i

    def union(self, i, j):
        ri, rj = self.find(i), self.find(j)
        if ri != rj:
            self.p[max(ri, rj)] = min(ri, rj)


def load_figshare(mat_dir):
    """Cheng et al. figshare .mat files (MATLAB v7.3 / HDF5): image, PID, label, tumorMask."""
    import h5py

    out, skipped = [], []
    for f in sorted(glob.glob(os.path.join(mat_dir, "**", "*.mat"), recursive=True)):
        try:
            h = h5py.File(f, "r")
        except OSError:  # not MATLAB v7.3
            skipped.append(os.path.basename(f))
            continue
        with h:
            if "cjdata" not in h:  # e.g. cvind.mat, the cross-validation indices
                skipped.append(os.path.basename(f))
                continue
            cj = h["cjdata"]
            img = np.array(cj["image"], dtype=np.float64)
            pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).ravel())
            mask = np.array(cj["tumorMask"], dtype=np.uint8)
            lab = int(np.array(cj["label"]).ravel()[0])
        u8 = ((img - img.min()) / (np.ptp(img) + 1e-8) * 255).astype(np.uint8)
        out.append({"mat": f, "pid": pid, "fig_label": lab, "u8": u8, "mask": mask})
    if skipped:
        print(f"figshare: skipped {len(skipped)} .mat file(s) that are not image slices: {skipped[:5]}", flush=True)
    return out


def attach_figshare(df, H, Z, fig, radius, ncc_thr, out_dir):
    """Match images to figshare slices; add patient id and an aligned tumour mask where found.

    Z: z-scored thumbnails of the Kaggle images, from zscore().
    """
    sigs = [signature(r["u8"]) for r in fig]
    HF = np.stack([s[0] for s in sigs])
    ZF = zscore([s[1] for s in sigs])
    cand = hash_matches(H, HF, radius)
    print(f"figshare: {len(fig):,} slices, {len(cand):,} candidate pairs", flush=True)
    scores, tr = ncc_pairs(Z, ZF, cand, label="figshare NCC")
    best = {}
    for (i, j), s, t in zip(cand, scores, tr):
        i, j, s, t = int(i), int(j), float(s), int(t)
        if s >= ncc_thr and s > best.get(i, (0, None, None))[0]:
            best[i] = (s, j, t)
    os.makedirs(os.path.join(out_dir, "masks"), exist_ok=True)
    df["patient"], df["mask_path"] = None, None
    for i, (s, j, t) in best.items():
        df.at[i, "patient"] = "fig_" + fig[j]["pid"]
        # thumbnails matched as A[i] ~ dihedral(B[j])[t]; apply the same transform to the mask
        m = dihedral(fig[j]["mask"])[t]
        w, h = int(df.at[i, "width"]), int(df.at[i, "height"])
        m = np.asarray(Image.fromarray(m * 255).resize((w, h), Image.NEAREST))
        p = os.path.join(out_dir, "masks", f"{i:05d}.png")
        Image.fromarray(m).save(p)
        df.at[i, "mask_path"] = p
    return df


def metadata_probe(df, families):
    """Balanced accuracy of a classifier that sees no anatomy, under family-grouped CV."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict

    header = pd.get_dummies(df[["width", "height", "file_bytes", "mode", "fmt"]], columns=["mode", "fmt"]).astype(float)
    header["aspect"] = df["width"] / df["height"]
    plus = header.assign(mean=df["mean"], std=df["std"], black_frac=df["black_frac"], border_mean=df["border_mean"])
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    res = {}
    for task, y in {"4class": df["label"].values, "tumour_vs_none": (df["label"] != "notumor").values}.items():
        for name, X in {"header_only": header, "header_plus_intensity": plus}.items():
            pred = cross_val_predict(HistGradientBoostingClassifier(random_state=SEED), X.values, y, groups=families, cv=cv)
            res[f"{task}/{name}"] = round(float(balanced_accuracy_score(y, pred)), 4)
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--out", default="audit")
    ap.add_argument("--figshare", default=None, help="folder with the figshare .mat files (optional)")
    ap.add_argument("--radius", type=int, default=8, help="max Hamming distance for a pHash candidate")
    ap.add_argument("--ncc", type=float, default=0.98,
                    help="NCC needed to confirm a near-duplicate; sweep 0.90-0.995 and inspect pairs near the cut")
    ap.add_argument("--ncc-floor", type=float, default=0.90,
                    help="candidate pairs scoring at least this are saved to candidates.csv for threshold sweeps")
    ap.add_argument("--folds", type=int, default=5)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    df = list_like_tpal(args.root)
    df["tpal_split"] = tpal_split(df)
    print(f"{len(df)} images; T-PAL split sizes:", df["tpal_split"].value_counts().to_dict(), flush=True)

    metas, H, smalls = [], [], []
    for n, p in enumerate(df["image_path"], 1):
        m, h, a = describe(p)
        metas.append(m)
        H.append(h)
        smalls.append(a)
        if n % 1000 == 0 or n == len(df):
            print(f"  hashed {n:,}/{len(df):,} images", flush=True)
    df = pd.concat([df, pd.DataFrame(metas)], axis=1)
    H = np.stack(H)
    Z = zscore(smalls)
    del smalls

    uf = UnionFind(len(df))
    for key in ("md5_file", "md5_pixels"):
        for grp in df.groupby(key).groups.values():
            grp = list(grp)
            for j in grp[1:]:
                uf.union(grp[0], j)
    cand = hash_matches(H, H, args.radius)
    cand = np.unique(np.sort(cand[cand[:, 0] != cand[:, 1]], 1), axis=0)
    print(f"{len(cand):,} candidate pairs within pHash radius {args.radius}; confirming with NCC", flush=True)
    scores, _ = ncc_pairs(Z, Z, cand, label="NCC")
    keep = scores >= args.ncc_floor
    cands = pd.DataFrame({"i": cand[keep, 0], "j": cand[keep, 1], "ncc": np.round(scores[keep], 4)})
    cands["a"], cands["b"] = df.image_path.values[cands.i], df.image_path.values[cands.j]
    cands["label_a"], cands["label_b"] = df.label.values[cands.i], df.label.values[cands.j]
    cands.to_csv(f"{args.out}/candidates.csv", index=False)
    conf = cands[cands.ncc >= args.ncc]
    for i, j in zip(conf.i, conf.j):
        uf.union(int(i), int(j))
    conf[["a", "b", "label_a", "label_b", "ncc"]].to_csv(f"{args.out}/pairs.csv", index=False)
    pairs = conf

    summary = {"phash_radius": args.radius, "ncc_threshold": args.ncc, "candidate_pairs_checked": int(len(cand)),
               f"candidate_pairs_ncc_ge_{args.ncc_floor}": int(len(cands))}
    if args.figshare:
        df = attach_figshare(df, H, Z, load_figshare(args.figshare), args.radius, args.ncc, args.out)
        for _, grp in df.dropna(subset=["patient"]).groupby("patient").groups.items():
            grp = list(grp)
            for j in grp[1:]:
                uf.union(grp[0], j)
        summary["figshare_matched_images"] = int(df["patient"].notna().sum())
        summary["figshare_patients"] = int(df["patient"].nunique())

    df["family"] = [uf.find(i) for i in range(len(df))]
    fam_splits = df.groupby("family")["tpal_split"].agg(set)
    fam_labels = df.groupby("family")["label"].agg(set)
    df["family_in_train"] = df["family"].map(lambda f: "train" in fam_splits[f])
    df["family_size"] = df.groupby("family")["family"].transform("size")
    summary.update({
        "images": len(df),
        "byte_identical_extra_copies": int(len(df) - df["md5_file"].nunique()),
        "pixel_identical_extra_copies": int(len(df) - df["md5_pixels"].nunique()),
        "confirmed_near_duplicate_pairs": len(pairs),
        "families": int(df["family"].nunique()),
        "images_in_families_of_size_gt1": int((df["family_size"] > 1).sum()),
        "families_with_conflicting_labels": int((fam_labels.map(len) > 1).sum()),
    })
    for s in ("val", "test"):
        sub = df[df["tpal_split"] == s]
        summary[f"tpal_{s}_with_train_family_member"] = f"{int(sub.family_in_train.sum())}/{len(sub)} ({sub.family_in_train.mean():.1%})"
        summary[f"tpal_{s}_leak_by_class"] = sub.groupby("label")["family_in_train"].mean().round(3).to_dict()
    fam_folders = df.groupby("family")["folder"].agg(set)
    official = df.loc[df["folder"] == "Testing", "family"].map(lambda f: "Training" in fam_folders[f])
    summary["official_test_with_training_family_member"] = f"{int(official.sum())}/{len(official)} ({official.mean():.1%})"
    summary["metadata_probe_balanced_accuracy"] = metadata_probe(df, df["family"].values)

    from sklearn.model_selection import StratifiedGroupKFold
    df["fold"] = -1
    sgkf = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)
    for k, (_, te) in enumerate(sgkf.split(df, df["label"], groups=df["family"])):
        df.loc[df.index[te], "fold"] = k
    summary["fold_sizes"] = df["fold"].value_counts().sort_index().to_dict()

    df.to_csv(f"{args.out}/images.csv", index=False)
    with open(f"{args.out}/summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=str)
    print(json.dumps(summary, indent=2, default=str))


if __name__ == "__main__":
    main()

## 2. Find the dataset and check it matches the T-PAL notebook

In [ ]:
def find_root(base="/kaggle/input"):
    if os.environ.get("AUDIT_ROOT"):
        return os.environ["AUDIT_ROOT"]
    for d, subdirs, _ in os.walk(base):
        if {"Training", "Testing"} <= set(subdirs):
            return d
    raise FileNotFoundError("No folder with Training/ and Testing/ under /kaggle/input. "
                            "Add the dataset masoudnickparvar/brain-tumor-mri-dataset with Add Input.")

ROOT = find_root()
print("Dataset folder:", ROOT)

sys.path.insert(0, os.getcwd())
import leakage_audit as la

df = la.list_like_tpal(ROOT)
df["tpal_split"] = la.tpal_split(df)
print(f"{len(df):,} images\n")
print(df.groupby(["label", "folder"]).size().unstack())
sizes = df["tpal_split"].value_counts().to_dict()
checks = {
    "7,200 images (T-PAL PDF p.2)": len(df) == 7200,
    "1,800 images per class (T-PAL PDF p.2)": set(df["label"].value_counts()) == {1800},
    "split 5,040 / 1,080 / 1,080 (T-PAL PDF p.20)": (sizes.get("train"), sizes.get("val"), sizes.get("test")) == (5040, 1080, 1080),
    "first 10 rows are pituitary, as listed on T-PAL PDF p.2": df["label"].head(10).tolist() == ["pituitary"] * 10,
}
print()
for name, ok in checks.items():
    print("OK      " if ok else "MISMATCH", name)
if not all(checks.values()):
    print("\nWARNING: this listing differs from the T-PAL notebook's. The dataset version may differ, "
          "so the rebuilt split may not be exactly T-PAL's.")

## 3. Optional: original figshare files (patient IDs and tumour masks)
The figshare set (Cheng et al., 2015) holds 3,064 slices from 233 patients, with tumour masks. Matching
Kaggle images to it merges slices of one patient into one family, and gives aligned masks for the
lesion-evidence loss and for scoring explanations.

In [ ]:
import hashlib, urllib.request, zipfile

def md5sum(path, block=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

if USE_FIGSHARE and not FIG_DIR:
    FIG_DIR = os.path.join(WORK, "figshare")
    os.makedirs(FIG_DIR, exist_ok=True)
    files = json.load(urllib.request.urlopen("https://api.figshare.com/v2/articles/1512427/files"))
    for f in files:
        if not f["name"].endswith(".zip"):
            continue
        dst = os.path.join(FIG_DIR, f["name"])
        print(f"downloading {f['name']} ({f['size'] / 1e6:.0f} MB)", flush=True)
        urllib.request.urlretrieve(f["download_url"], dst)
        assert md5sum(dst) == f["computed_md5"], f"MD5 mismatch for {f['name']}: delete it and run this cell again"
        with zipfile.ZipFile(dst) as z:
            z.extractall(FIG_DIR)
        os.remove(dst)

if FIG_DIR:
    import h5py  # preinstalled on Kaggle; reads the MATLAB v7.3 files
    n_mat = len(glob.glob(os.path.join(FIG_DIR, "**", "*.mat"), recursive=True))
    print(f"{n_mat:,} figshare .mat files in {FIG_DIR} (the full set has 3,064)")
else:
    print("figshare step skipped: patient IDs and tumour masks will not be recovered")

## 4. Run the audit
Progress is printed as it goes. Hashing the 7,200 images takes a few minutes. The near-duplicate check
then compares candidate pairs, at roughly 25 seconds per million pairs on a fast laptop CPU (expect
slower on Kaggle). The number of candidate pairs is printed first.

In [ ]:
cmd = [sys.executable, "leakage_audit.py", "--root", ROOT, "--out", OUT,
       "--ncc", str(NCC), "--ncc-floor", str(NCC_FLOOR)]
if FIG_DIR:
    cmd += ["--figshare", FIG_DIR]
print(" ".join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "The audit failed; see the messages above."

## 5. Results

In [ ]:
import numpy as np, pandas as pd

S = json.load(open(os.path.join(OUT, "summary.json")))
img = pd.read_csv(os.path.join(OUT, "images.csv"))
pairs = pd.read_csv(os.path.join(OUT, "pairs.csv"))
cands = pd.read_csv(os.path.join(OUT, "candidates.csv"))

print(f"{S['images']:,} images in {S['families']:,} families; "
      f"{S['images_in_families_of_size_gt1']:,} images have at least one copy or twin")
print(f"exact copies: {S['byte_identical_extra_copies']} byte-identical, {S['pixel_identical_extra_copies']} pixel-identical")
print(f"candidate pairs checked: {S['candidate_pairs_checked']:,}; near-duplicate pairs at NCC >= {NCC}: "
      f"{S['confirmed_near_duplicate_pairs']:,}")
print(f"families whose images carry different labels: {S['families_with_conflicting_labels']}")
if "figshare_matched_images" in S:
    print(f"figshare: {S['figshare_matched_images']:,} images matched to {S['figshare_patients']} patients")

### Table 1 (draft): images with a family member in training
For the official split, "training" is the *Training* folder. For T-PAL's split it is T-PAL's training
set. The 95% intervals are Wilson intervals per image.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def leak_table(families):
    fam = pd.Series(np.asarray(families))
    rows = []
    for name, sel, ref in [("Official Testing folder", img.folder == "Testing", img.folder == "Training"),
                           ("T-PAL validation split", img.tpal_split == "val", img.tpal_split == "train"),
                           ("T-PAL test split", img.tpal_split == "test", img.tpal_split == "train")]:
        sel, ref = sel.values, ref.values
        leaked = fam[sel].isin(set(fam[ref])).values
        k, n = int(leaked.sum()), int(sel.sum())
        lo, hi = wilson(k, n)
        row = {"set": name, "images": n, "with a family member in training": k,
               "%": round(100 * k / max(n, 1), 1), "95% CI": f"{100 * lo:.1f}-{100 * hi:.1f}"}
        for c, v in pd.Series(leaked).groupby(img.label.values[sel]).mean().items():
            row[f"% {c}"] = round(100 * v, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("set")

table1 = leak_table(img.family)
table1.to_csv(os.path.join(OUT, "table1_leakage.csv"))
table1

### How the numbers depend on the near-duplicate threshold
Families are rebuilt at each NCC threshold from the saved candidate pairs. Exact copies and figshare
patients are always merged. Report this sweep in the supplement.

In [ ]:
def families_at(t):
    uf = la.UnionFind(len(img))
    keys = ["md5_file", "md5_pixels"] + (["patient"] if "patient" in img else [])
    for key in keys:
        for grp in img.dropna(subset=[key]).groupby(key).groups.values():
            grp = list(grp)
            for j in grp[1:]:
                uf.union(grp[0], j)
    for i, j in cands.loc[cands.ncc >= t, ["i", "j"]].itertuples(index=False):
        uf.union(int(i), int(j))
    return [uf.find(i) for i in range(len(img))]

rows = []
for t in [0.90, 0.92, 0.95, 0.98, 0.99, 0.995]:
    if t < NCC_FLOOR:
        continue
    tb = leak_table(families_at(t))
    rows.append({"NCC threshold": t, "pairs": int((cands.ncc >= t).sum()),
                 **{f"{s} %": tb.loc[s, "%"] for s in tb.index}})
sweep = pd.DataFrame(rows).set_index("NCC threshold")
sweep.to_csv(os.path.join(OUT, "threshold_sweep.csv"))
if NCC in sweep.index:
    same = sweep.loc[NCC, "T-PAL test split %"] == table1.loc["T-PAL test split", "%"]
    print("Row", NCC, "matches Table 1:", same)
sweep

### Look at the pairs
Check by eye that the pairs are true copies. Each column shows image A, image B after the flip or
rotation that matches it best, and an overlay (A in red, B in green).
- **Yellow structures with no colour fringes:** the same scan. A brightness or contrast change only
  tints the overlay orange or green.
- **Red and green fringes along edges:** a different slice. Neighbouring slices of one patient look
  like this.

Small thumbnails side by side are misleading: processed copies (sharpened, brightened) can look like
different scans until they are overlaid. Look hardest at pairs just above the threshold. Red titles
mark pairs whose labels differ.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def load01(p, n=256):
    a = np.asarray(Image.open(p).convert("L").resize((n, n), Image.BILINEAR), dtype=np.float32)
    return (a - a.min()) / (np.ptp(a) + 1e-6)

def align(A, B):
    # the flip or rotation of B that correlates best with A
    best, out = -2.0, B
    for k in range(4):
        r = np.rot90(B, k)
        for t in (r, r[:, ::-1]):
            s = np.corrcoef(A.ravel(), t.ravel())[0, 1]
            if s > best:
                best, out = s, t
    return out

def show_pairs(d, title, n=8):
    d = d.head(n)
    if d.empty:
        print("No pairs:", title)
        return
    fig, ax = plt.subplots(3, len(d), figsize=(2.4 * len(d), 8.2), squeeze=False)
    for k, r in enumerate(d.itertuples()):
        A = load01(r.a)
        B = align(A, load01(r.b))
        color = "red" if r.label_a != r.label_b else "black"
        ax[0, k].imshow(A, cmap="gray")
        ax[0, k].set_title(f"NCC {r.ncc:.3f}\n{r.label_a} · {os.path.basename(r.a)}", fontsize=7, color=color)
        ax[1, k].imshow(B, cmap="gray")
        ax[1, k].set_title(f"{r.label_b} · {os.path.basename(r.b)}", fontsize=7, color=color)
        ax[2, k].imshow(np.dstack([A, B, np.zeros_like(A)]))
        ax[2, k].set_title("overlay (red A, green B)", fontsize=7)
        for a in ax[:, k]:
            a.axis("off")
    fig.suptitle(title)
    plt.tight_layout(h_pad=1.5)
    plt.show()

show_pairs(pairs.sort_values("ncc"), f"Confirmed pairs closest to the threshold (NCC >= {NCC})")
show_pairs(cands[cands.ncc < NCC].sort_values("ncc", ascending=False), f"Pairs just below the threshold")
show_pairs(pairs[pairs.label_a != pairs.label_b], "Confirmed pairs with different labels")

### Families whose images carry different labels

In [ ]:
n_labels = img.groupby("family")["label"].nunique()
conflict = img[img.family.isin(n_labels[n_labels > 1].index)].sort_values("family")
print(f"{conflict.family.nunique()} families, {len(conflict)} images")
conflict[["family", "label", "folder", "tpal_split", "image_path"]].head(40)

### Shortcut check: do file details alone predict the label?
A classifier sees only image size, file size and format ("header only"), or those plus simple
intensity statistics. It never sees the anatomy, and it is scored with family-grouped
cross-validation. Scores well above chance mean the label can be guessed from the image source.

In [ ]:
probe = pd.Series(S["metadata_probe_balanced_accuracy"]).rename("balanced accuracy").to_frame()
probe["chance"] = [0.25 if k.startswith("4class") else 0.5 for k in probe.index]
display(probe)
size = img.width.astype(str) + "x" + img.height.astype(str)
top = size.value_counts().head(8).index
pd.crosstab(img.label, size.where(size.isin(top), "other"))

## 6. Leakage-free folds for re-training

In [ ]:
img["rel_path"] = [os.path.relpath(p, ROOT) for p in img.image_path]
cols = ["rel_path", "label", "folder", "family", "fold", "tpal_split"] + [c for c in ("patient", "mask_path") if c in img]
img[cols].to_csv(os.path.join(OUT, "grouped_folds.csv"), index=False)
assert img.groupby("family")["fold"].nunique().max() == 1, "a family spans two folds"
print("No family spans two folds.\n")
pd.crosstab(img.fold, img.label, margins=True)

**Using the folds in the T-PAL notebook.** Save a version of this notebook, then in the T-PAL
notebook use **Add Input → Your Work** to add this notebook's output. Then replace the random split
with grouped folds, for example with fold `k` as test and fold `k + 1` as validation:

```python
folds = pd.read_csv("/kaggle/input/<this-notebook>/audit/grouped_folds.csv")
df["rel_path"] = [os.path.relpath(p, BASE_DIR) for p in df.image_path]
df = df.merge(folds[["rel_path", "fold", "family"]], on="rel_path", how="left", validate="one_to_one")
k = 0
te_idx = np.where(df.fold == k)[0]
va_idx = np.where(df.fold == (k + 1) % 5)[0]
tr_idx = np.where(~df.fold.isin([k, (k + 1) % 5]))[0]
```

Run all five values of `k` (and three seeds each) and report the mean and spread. Flip augmentation
is safe once the split is grouped by family.

In [ ]:
import shutil
zip_path = shutil.make_archive(os.path.join(WORK, "audit"), "zip", OUT)
print("Saved", zip_path)
print("Files:", sorted(os.listdir(OUT)))

## What to bring back
- `table1_leakage.csv`: the T-PAL test row is the headline number for the paper.
- `threshold_sweep.csv`: for the supplement.
- Whether the example pairs look like true copies.
- The shortcut-check scores.
- The dataset version number from the Input panel, and today's date.